
# 12 WT-plus 单因子：从资料入库到验证报告

> **本章要解决的问题**：一篇研报里的一句话，怎样变成一个可计算、可复查、可冻结的因子？WT-plus 的队列、准入、计算、验证、报告，各自在做什么？

## 本章导读

第01章讲了一笔交易，第03章讲了一份报告。这一章把“报告”放进 WT-plus 的流水线：

```
资料 → 研究提案 → 因子卡 → 表达式 → 计算快照 → 验证报告 → 冻结/观察
```


**WT-plus 是什么**：WT 是你改造后的 WonderTrader——C++ 核心 [WonderTrader-plus](https://github.com/newbigdeng/WonderTrader-plus)，Python 封装 [wtpy-plus](https://github.com/newbigdeng/wtpy-plus)。它负责撮合、回放、账户/持仓；因子研究由 RD-Agent、KunQuant、TA-Lib、Alphalens、fincore 和护栏完成（见第37章）。

每一步都有输入、输出、失败模式。看懂这条链，才能判断“报告为空”是数据问题、表达式问题，还是验证没过。

本章分六节：

1. 资料入库与因子卡；
2. 从一句话到表达式；
3. 计算快照与未来函数；
4. 验证报告：覆盖率、IC、分组、成本；
5. 状态机：提案/计算/验证/冻结；
6. WT-plus 工具链：每个工具只回答一个问题。

### 学习目标

- 给定资料和字段，能填一张因子卡，并判断“能不能算”。
- 能把一句话翻译成表达式，并和手算对齐。
- 能做快照和未来扰动检查。
- 能生成覆盖率、IC、分组、成本四块验证报告。
- 能按状态机判断提案/计算/验证/冻结。

### 先修

第01章的因子/标签；第03章的 IC/分组/成本；会 pandas。

---

### 环境准备



## 本章知识结构图

<svg style="max-width:100%;height:auto" xmlns="http://www.w3.org/2000/svg" width="800" height="450" viewBox="0 0 800 450" role="img" aria-label="12 WT-plus 单因子：从资料入库到验证报告" font-family="Microsoft YaHei, SimHei, sans-serif">
<title>12 WT-plus 单因子：从资料入库到验证报告｜知识图</title>
<desc>本章主线：资料 → 因子卡 → 表达式 → 计算快照 → 验证报告 → 冻结观察，每一步都可复查。</desc>
<rect x="0" y="0" width="800" height="450" fill="#FFFFFF"/>
<defs><marker id="a11" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#1B3A5C"/></marker><marker id="ao11" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#E07B39"/></marker><marker id="ag11" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 Z" fill="#6E7B8B"/></marker></defs>
<rect x="40" y="19" width="46" height="29" fill="#E07B39" rx="8"/>
<text x="63" y="39" font-size="15" fill="#FFFFFF" text-anchor="middle" font-weight="bold">12</text>
<text x="98" y="41" font-size="22" fill="#1B3A5C" font-weight="bold">WT-plus 单因子：从资料入库到验证报告</text>
<line x1="40" y1="57" x2="760" y2="57" stroke="#E6EFF7" stroke-width="2" stroke-linecap="round"/>
<line x1="40" y1="407" x2="760" y2="407" stroke="#E6EFF7" stroke-width="1.6" stroke-linecap="round"/>
<rect x="40" y="418" width="4" height="10" fill="#E07B39" rx="1"/>
<text x="52" y="428" font-size="13" fill="#6E7B8B">本章主线：资料 → 因子卡 → 表达式 → 计算快照 → 验证报告 → 冻结观察，每一步都可复查。</text>
<text x="42" y="90" font-size="10" fill="#6E7B8B">WT-plus 流水线：队列 · 幂等 · PIT 快照 · 版本 manifest</text>
<circle cx="106" cy="108" r="9" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.4"/>
<text x="106" y="111.5" font-size="9" fill="#1B3A5C" text-anchor="middle" font-weight="bold">1</text>
<rect x="42" y="120" width="128" height="92" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="106" y="158.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">资料入库</text>
<text x="106" y="177.4" font-size="9" fill="#6E7B8B" text-anchor="middle">字段、合约规则、来源</text>
<circle cx="252" cy="108" r="9" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.4"/>
<text x="252" y="111.5" font-size="9" fill="#1B3A5C" text-anchor="middle" font-weight="bold">2</text>
<rect x="188" y="120" width="128" height="92" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="252" y="158.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">因子卡</text>
<text x="252" y="177.4" font-size="9" fill="#6E7B8B" text-anchor="middle">一句话 → 可计算的表达式</text>
<circle cx="398" cy="108" r="9" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.4"/>
<text x="398" y="111.5" font-size="9" fill="#1B3A5C" text-anchor="middle" font-weight="bold">3</text>
<rect x="334" y="120" width="128" height="92" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="398" y="158.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">表达式 + 快照</text>
<text x="398" y="177.4" font-size="9" fill="#6E7B8B" text-anchor="middle">PIT、无未来函数</text>
<circle cx="544" cy="108" r="9" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.4"/>
<text x="544" y="111.5" font-size="9" fill="#1B3A5C" text-anchor="middle" font-weight="bold">4</text>
<rect x="480" y="120" width="128" height="92" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="544" y="152" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">验证报告</text>
<text x="544" y="170.6" font-size="9" fill="#6E7B8B" text-anchor="middle">覆盖 / IC / 分组 / 成</text>
<text x="544" y="184.1" font-size="9" fill="#6E7B8B" text-anchor="middle">本</text>
<circle cx="690" cy="108" r="9" fill="#E07B39" stroke="#E07B39" stroke-width="1.4"/>
<text x="690" y="111.5" font-size="9" fill="#FFFFFF" text-anchor="middle" font-weight="bold">5</text>
<rect x="626" y="120" width="128" height="92" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.5" rx="14"/>
<text x="690" y="158.8" font-size="12.5" fill="#1B3A5C" text-anchor="middle" font-weight="bold">冻结观察</text>
<text x="690" y="177.4" font-size="9" fill="#6E7B8B" text-anchor="middle">前向达标再上线</text>
<path d="M 170 166 L 188 166" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a11)"/>
<path d="M 316 166 L 334 166" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a11)"/>
<path d="M 462 166 L 480 166" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a11)"/>
<path d="M 608 166 L 626 166" fill="none" stroke="#1B3A5C" stroke-width="1.7" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#a11)"/>
<rect x="42" y="236" width="712" height="46" fill="#FFFFFF" stroke="#E07B39" stroke-width="1.3" rx="14" stroke-dasharray="6 4"/>
<text x="58" y="254" font-size="10" fill="#E07B39" font-weight="bold">常见断点</text>
<text x="58" y="271" font-size="9.5" fill="#6E7B8B">资料 ≠ 因子 · 快照用到未来数据 · 报告为空还继续 · 冻结条件没写</text>
<text x="42" y="304" font-size="10" fill="#6E7B8B">验证报告的四个数</text>
<rect x="42" y="312" width="160" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="122" y="337" font-size="11.5" fill="#1B3A5C" text-anchor="middle">覆盖率</text>
<rect x="222" y="312" width="160" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="302" y="337" font-size="11.5" fill="#1B3A5C" text-anchor="middle">IC / ICIR</text>
<rect x="402" y="312" width="160" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="482" y="337" font-size="11.5" fill="#1B3A5C" text-anchor="middle">分组收益</text>
<rect x="582" y="312" width="160" height="42" fill="#E6EFF7" stroke="#1B3A5C" stroke-width="1.3" rx="14"/>
<text x="662" y="337" font-size="11.5" fill="#1B3A5C" text-anchor="middle">换手与成本</text>
</svg>

> **图**：本章主线：资料 → 因子卡 → 表达式 → 计算快照 → 验证报告 → 冻结观察，每一步都可复查。


In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：因子研究员、数据工程、WT 流水线维护。
- **输入**：资料/字段、合约规则、表达式。
- **输出**：因子卡、覆盖率、IC/分组/成本验证报告。
- **在流程的哪一步**：WT 单因子流水线：入库 → 计算 → 验证 → 冻结。
- **失败模式**：把“资料”当“因子”、快照用到未来数据、报告为空还继续。
- **最小工作流**：资料入库 → 因子卡 → 表达式/手算对齐 → 快照/扰动 → 四块验证 → 状态机。

---
## 1. 资料入库与因子卡

### 1.1 资料不是因子

一篇研报说“放量上涨强势”，这句话还不能计算。要拆成：

- 观察谁（品种/合约）；
- 用哪些字段（成交量、收盘价）；
- 什么窗口（前 3 根/前 20 日）；
- 输出方向（越高越看多/看空）；
- 何时可用（收盘后/开盘前）。

**资料准入**的意思是：先判断“本地有没有这些字段、能不能复现”，再谈因子。

### 1.2 因子卡

一张最小因子卡：

| 字段 | 例子 | 作用 |
|---|---|---|
| name | volume_ratio | 唯一名称 |
| expression | volume / mean(volume, 3) | 计算式 |
| inputs | volume | 依赖字段 |
| window | 3 | 回看长度 |
| direction | +1 | 分数越高越看多 |
| known_at | bar_close | 何时可见 |
| missing | skip | 缺失处理 |

没有因子卡，三个月后没人能复现你的结果。

### 1.3 代码：因子卡校验



In [2]:

card = {
    "name": "volume_ratio",
    "expression": "volume / mean(volume, 3)",
    "inputs": {"volume"},
    "window": 3,
    "direction": 1,
    "known_at": "bar_close",
    "missing": "skip",
}

available = {"close", "volume", "open_interest"}
missing = card["inputs"] - available
print("可计算" if not missing else f"缺字段 {sorted(missing)}")
print("方向", "看多" if card["direction"] > 0 else "看空")


可计算
方向 看多



**读数与应用**：因子卡校验先看“字段够不够”，再看“方向/可用时刻”。字段不够，不要硬造；换一个可复现的定义，或在报告里注明“与原文不同”。

**常见错误**：

1. 只写“放量”，不写基准窗口和比较方式。
2. 不写方向，后面的人不知道高分代表什么。
3. 不写可用时刻，回测时偷看未来。

**停下来想一想**：原文用“盘口撤单量”，本地只有 OHLCV，能不能复现？

<details><summary>对照你的思路</summary>

不能按原定义复现。成交量无法唯一反推撤单量；可以另立一个“分钟成交量/前 3 分钟均量”的假设，但要在因子卡里注明“与原文不同”，不能宣称复现了原文。

</details>

---



## 2. 从一句话到表达式

### 2.1 表达式要素

- 字段：open/high/low/close/volume/open_interest；
- 算子：shift、rolling、mean、std、rank、corr；
- 窗口：前 3 根、前 20 日；
- 对齐：按交易日/交易节，不跨休市；
- 缺失：预热不足返回空，不填 0。

### 2.2 例子

“本根成交量 / 前两根平均量”：

```text
volume / mean(volume, 2).shift(1)
```

注意 `shift(1)`：只能用前两根，不能用当根。

### 2.3 手算

三根已结束的量：10、30、40。

- 当根 40 的基准：前两根均值 $(10+30)/2=20$；
- 分数 $40/20=2$。

如果把当根也算进基准，$(30+40)/2=35$，分数 $40/35=1.14$，回答的是另一个问题。

**公式拆解**
- **符号**：$v_t$ 第 $t$ 根已结束成交量（手），$\bar v_{t-1}=(v_{t-1}+v_{t-2})/2$ 前两根均量，时间顺序 $t-2<t-1<t$。
- **结构**：$s_t=v_t/\bar v_{t-1}$ 是倍数：分子为当根、分母为前段时间，`shift(1)` 保证分母不含当根。
- **代入**：10、30、40、20 → $40/20=2.0$、$20/35\approx0.571$。
- **边界**：分母为 0（停牌/无量）或预热不足 → `NaN`，不能填 0。
- **用途**：把“放量”变成可排序分数；不是上涨概率，也不代表流动性。

### 2.4 代码：表达式与手算对齐



In [3]:

import pandas as pd

v = pd.Series([10.0, 30, 40, 20])
ref = v.shift(1).rolling(2).mean()
score = v / ref
print(pd.DataFrame({"volume": v, "ref": ref, "score": score}))


   volume   ref     score
0    10.0   NaN       NaN
1    30.0   NaN       NaN
2    40.0  20.0  2.000000
3    20.0  35.0  0.571429



**读数与应用**：第三根 40 的基准是 20，分数 2；第四根 20 的基准是 35，分数 0.571。前两根预热不足，返回 NaN。

**常见错误**：

1. 忘记 `shift(1)`，把当根算进基准。
2. 窗口跨休市，把 10:15 和 10:35 当连续。
3. 缺失填 0，把“没数据”当成“分数为 0”。

**停下来想一想**：为什么表达式要写“窗口 + 对齐 + 缺失”，而不是只写公式？

<details><summary>对照你的思路</summary>

同一个公式，窗口不同、对齐不同、缺失处理不同，结果会完全不同。因子卡要能让他人按同样规则复算；只写“成交量比”没有信息量，无法复查。

</details>

---



## 3. 计算快照与未来函数

### 3.1 快照

计算因子时，要保存：

- 数据快照日期/版本；
- 表达式版本；
- 字段映射；
- 输出文件/表名。

否则供应商修订历史后，同一表达式会算出不同结果。

### 3.2 未来扰动

复制输入，把 $t$ 之后的数据改掉，重新算 $t$ 时刻的因子。若因子变了，说明读了未来。

### 3.3 代码：扰动检查



In [4]:

import pandas as pd

for last in [105.0, 999.0]:
    s = pd.Series([100.0, 101.0, last])
    trailing = s.rolling(2).mean()[1]
    centered = s.rolling(3, center=True).mean()[1]
    print("最后一根", last, "右对齐", trailing, "居中", centered)


最后一根 105.0 右对齐 100.5 居中 102.0
最后一根 999.0 右对齐 100.5 居中 400.0



**读数与应用**：右对齐均线在 $t=1$ 时只用 100、101，最后一根改成 999 也不变；居中都线用了最后一根，从 102 变成 400。居中窗口是研究工具，不能做信号。

**常见错误**：

1. 只扰动价格，不扰动成交量/持仓量。
2. 特征正确，但标签/执行用了未来。
3. 数据源本身已倒填修订值，扰动行不一定发现。

**停下来想一想**：为什么“居中均线”在回测里是未来函数？

<details><summary>对照你的思路</summary>

居中均线在 $t$ 时刻的窗口包含 $t+1,t+2$ 的数据；在 $t$ 时刻你还没有这些数据。它适合事后平滑/画图，不适合生成交易信号。

</details>

---



## 4. 验证报告：覆盖率、IC、分组、成本

### 4.1 覆盖率

每天能算出因子的标的比例。缺失可能集中在：

- 新合约/刚上市；
- 停牌/无成交；
- 夜盘/休市；
- 字段缺失。

覆盖率低，IC 再高也不可信。报告要同时给“每天有效标的数”和“缺失原因”。

### 4.2 IC 与分组

- 逐日 Rank IC 的均值、标准差、t 值、ICIR；
- 3/5 组收益、单调性、高减低；
- 每组样本数、独立交易日。

**公式拆解**
- **符号**：$s_{t,i}$ 分数、$r_{t,i}$ 未来收益；$IC_t$ 当日截面 Spearman；$\overline{IC}$、$s(IC)$ 均值/标准差，$T$ 交易日数。
- **结构**：$ICIR=\overline{IC}/s(IC)$（平均能力÷日间波动）；$t=ICIR\sqrt T$。
- **代入**：$T=60$、$IC=0.1109$、$s=0.2073$ → $ICIR\approx0.535$、$t\approx4.14$。
- **边界**：$s(IC)=0$ 无定义；标签重叠时 $t$ 被高估。
- **用途**：单因子初筛与比较；样本单位是“日”不是标的×日。

### 4.3 换手与成本

- 单边/双边换手；
- 手续费、价差、滑点；
- 扣费后的净收益。

### 4.4 代码：覆盖率



In [5]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(1101)
panel = pd.DataFrame(rng.normal(size=(50, 10)))
panel.iloc[:5, :3] = np.nan

coverage = panel.notna().mean()
print("各列覆盖率")
print(coverage.round(2))
print("日均有效标的", round(panel.notna().sum(axis=1).mean(), 2))


各列覆盖率
0    0.9
1    0.9
2    0.9
3    1.0
4    1.0
5    1.0
6    1.0
7    1.0
8    1.0
9    1.0
dtype: float64
日均有效标的 9.7



**读数与应用**：前三列前 5 行缺失，覆盖率 0.9；其他列 1.0。真实报告要按日看覆盖率，而不是只给一个全样本平均。

### 4.5 代码：最小验证报告



In [6]:

import numpy as np
import pandas as pd

rng = np.random.default_rng(1102)
T, N = 60, 20
score = pd.DataFrame(rng.normal(size=(T, N)))
future = 0.0008 * score + pd.DataFrame(rng.normal(0, 0.01, size=(T, N)))

ic = score.rank(axis=1).corrwith(future.rank(axis=1), axis=1)
groups = (((score.rank(axis=1) - 1) * 3) // N).astype(int)
group_ret = pd.DataFrame({g: future.where(groups == g).mean(axis=1) for g in range(3)})

print("IC 均值/标准差/t", round(ic.mean(), 4), round(ic.std(ddof=1), 4),
      round(ic.mean() / (ic.std(ddof=1) / np.sqrt(T)), 3))
print("分组平均收益", group_ret.mean().round(5).to_dict())


IC 均值/标准差/t 0.1109 0.2073 4.142
分组平均收益 {0: -0.00158, 1: -0.00024, 2: 0.00115}



**读数与应用**：合成面板 IC 约 0.1，分组单调。真实报告还要加换手、成本、样本外；合成数据只用来校验代码。

**常见错误**：

1. 只报全样本覆盖率，不看逐日。
2. 分组不报样本数/独立交易日。
3. 只看 IC，不看扣费净收益。

**停下来想一想**：报告为空，先查哪一层？

<details><summary>对照你的思路</summary>

先查“数据/字段/日期”有没有覆盖，再查“表达式/窗口”是否算空，再查“验证/报告”是否失败。顺序是：输入 → 计算 → 验证 → 报告；不要一上来改模型。

</details>

---



## 5. 状态机：提案、计算、验证、冻结

### 5.1 四个状态

| 状态 | 含义 | 下一步 |
|---|---|---|
| proposed | 研究提案/因子卡 | 进入计算 |
| computed | 已算出分数/快照 | 进入验证 |
| validated | 有验证报告 | 冻结/观察 |
| frozen | 冻结，等待前向观察 | 观察/退役 |

状态不是“进度条”，而是“责任边界”：每个状态要有对应的 ID（提案 ID、job ID、报告 ID、版本 ID）。

### 5.2 失败模式

- 提案缺字段 → 停在准入；
- 计算 job 失败 → 看日志/字段/表达式；
- 验证报告为空 → 看覆盖率/日期/样本；
- 状态卡住 → 看队列/权限/依赖。

### 5.3 代码：状态转移



In [7]:

status = "computed"
allowed = {
    "proposed": {"computed"},
    "computed": {"validated"},
    "validated": {"frozen"},
    "frozen": set(),
}
print("当前", status, "可进入", allowed[status])


当前 computed 可进入 {'validated'}



**读数与应用**：状态机防止“跳过验证直接上线”。每个状态保存对应 ID，才能追溯“这份报告是哪次计算、哪个数据快照”。

**常见错误**：

1. 用“完成”一个词覆盖所有状态，出问题不知道卡在哪。
2. 不保存 job/report ID，无法复现。
3. 跳过验证直接冻结。

**停下来想一想**：为什么“计算完成”不等于“因子有效”？

<details><summary>对照你的思路</summary>

计算完成只说明分数算出来了；验证才回答“有没有排序能力、扣费后是否为正、样本外是否稳”。计算是生产，验证是质检；两件事不能合并。

</details>

---



## 6. WT-plus 工具链：每个工具只回答一个问题

### 6.1 WT-plus 是什么

WT 是你改造后的 WonderTrader：

- **WonderTrader-plus**：C++ 核心，负责撮合、回放、账户/持仓、执行语义；
- **wtpy-plus**：Python 封装，负责策略/数据/回放接口。

它回答“如果当时真的这么跑，账会怎样”，**不回答“这个因子有没有 alpha”**。因子研究由下面这条 Python 工具链完成。

### 6.2 一张表

| 环节 | 工具 | 回答的问题 | 产物 |
|---|---|---|---|
| 数据入口 | Great Expectations | 字段/时序/异常值是否合格 | 校验报告 |
| 研究循环 | RD-Agent | 假设→代码→回测→反馈 | 候选表达式 |
| 超参 | Optuna | 循环内哪组参数最好 | 参数/试验记录 |
| 计算 | KunQuant | 表达式能否批量编译加速 | 因子面板 |
| 基线 | TA-Lib | 新因子是否只是经典指标变体 | 基线因子 |
| 初筛 | Alphalens-reloaded | IC/ICIR/分组/换手 | 单因子报告 |
| 护栏 | DSR/PBO | 多轮搜索后是否过拟合 | 校正后显著性 |
| 组合 | fincore | 组合绩效/风险/归因 | 组合报告 |
| 追踪 | MLflow | 每轮能否复现 | 版本/参数/指标 |
| 回放 | WT-plus | 账户级执行会怎样 | 成交/费用/持仓 |

### 6.3 三个最容易被忽略的边界

1. **RD-Agent 的候选不是结论**：它扩大搜索，搜索越大，DSR/PBO 越不能省。
2. **KunQuant 快不等于算对**：它要求输入数量是 8 的倍数；仍要抽一行手算核对。
3. **Alphalens 初筛不等于组合可用**：过了 IC/分组，还要过 DSR/PBO、fincore 归因、WT-plus 回放。

### 6.4 最小流水线

```
Great Expectations → RD-Agent(+Optuna) → KunQuant/TA-Lib
→ Alphalens → DSR/PBO → fincore → WT-plus 回放
        ↑______________ MLflow ______________|
```

> **真实文件**：`factor_automation_worker.py`、`FactorToolchain.py`、`factor_compute.py`、`kunquant_runner.py`、`rd_factor_agent.py`、`factor_native_audit.py`；调用链与约束见第37章 §7。

### 6.5 报告为空时的排查顺序

1. 数据：日期/合约/字段有没有？Great Expectations 有没有报错？
2. 表达式：窗口是否吃掉全部样本？分母是否为 0？
3. 计算：KunQuant 输入长度是否补齐到 8 的倍数？
4. 验证：Alphalens 分组是否样本太少？
5. 报告：模板/权限/路径。

**常见错误**：

1. 把“RD-Agent 生成了表达式”当成“因子已生成”。
2. 把“KunQuant 算完”当成“算对”。
3. 不记录数据快照，供应商修订后无法解释差异。

**停下来想一想**：为什么同一表达式在不同数据快照上会算出不同结果？

<details><summary>对照你的思路</summary>

供应商会修订历史（补数据、改错误、调整合约映射）；快照不同，输入不同，结果就不同。要保存快照日期/版本，报告里注明“基于哪个快照”，否则无法复现。

</details>

---



## 7. 因子研究的四道门

单因子报告不是“IC 显著就通过”。按四道门检查：

1. **逻辑门**：经济机制是什么？失效条件是什么？
2. **样本门**：训练/测试/极端市场；样本外是否还成立？
3. **交易门**：换手、价差、冲击成本；扣费后还剩多少？
4. **风险门**：行业/市值/Beta/拥挤度暴露；中性化后是否消失？

四道门缺一不可。过不了交易门/风险门，IC 再高也不能进组合。

---

## 本章小结

### 一张表

| 环节 | 关键产物 | 检查 |
|---|---|---|
| 资料 | 摘要/候选 | 字段能否复现 |
| 因子卡 | 名称/表达式/方向/时刻 | 可复现 |
| 计算 | 分数/快照 | 未来函数 |
| 验证 | 覆盖率/IC/分组/成本 | 样本外 |
| 状态 | ID/状态 | 可追溯 |

### 八句话

1. 资料不是因子；先过“字段/可复现”准入。
2. 因子卡要写方向、窗口、可用时刻、缺失处理。
3. 表达式要写清 shift/对齐/缺失。
4. 计算要保存数据快照和表达式版本。
5. 未来扰动是检查泄漏的最快方法。
6. 报告为空，先查输入，再查计算，再查验证。
7. 计算完成不等于因子有效。
8. 状态/ID 是可追溯的底线。

---

## 本章习题

**1.** 写一张“成交量/前 3 日均量”的因子卡（名称、表达式、输入、方向、可用时刻）。

**2.** 三根量 10、30、40。用前两根均值算当根分数；如果把当根也算进去，分数是多少？

**3.** 右对齐均线和居中均线，哪个有未来函数？为什么？

**4.** 覆盖率 0.6，说明什么？报告还要看什么？

**5.** 计算 job 成功、验证报告为空，先查哪三层？

**6.** 为什么“研报摘要完成”不等于“因子已生成”？

**7.** 状态机里，computed 的下一步是什么？能不能直接 frozen？

**8.** 同一表达式在两个数据快照上结果不同，怎么解释？

<details><summary>参考答案</summary>

**1.** name=volume_ratio；expression=volume/mean(volume,3).shift(1)；inputs={volume}；direction=+1；known_at=bar_close。

**2.** 前两根均值 20，分数 2；含当根 $(30+40)/2=35$，分数 1.14。

**3.** 居中均线有未来函数：$t$ 时刻窗口包含 $t+1,t+2$ 的数据。

**4.** 40% 的标的/日期算不出因子；要看缺失原因、逐日覆盖率、是否集中在某些品种/时段。

**5.** 数据/字段/日期 → 表达式/窗口 → 验证/分组/报告。

**6.** 摘要只说明“有可读材料”；还要准入、表达式、计算、验证，才有因子/报告。

**7.** 下一步 validated；不能直接 frozen，除非显式允许跳过验证（不推荐）。

**8.** 数据快照/版本不同（修订、补数、合约映射），输入不同，结果不同；要记录快照。

</details>

---

## 参考资料

- WonderTrader-plus: https://github.com/newbigdeng/WonderTrader-plus
- wtpy-plus: https://github.com/newbigdeng/wtpy-plus
- RD-Agent: https://github.com/microsoft/RD-Agent
- KunQuant: https://github.com/Menooker/KunQuant
- Alphalens-reloaded: https://github.com/stefan-jansen/alphalens-reloaded
- fincore: https://github.com/cloudQuant/fincore
- 第37章（完整工具链与护栏）。
- 本项目 `F:\0_MyLog\_教材化\进度.md`（改造进度与构建命令）。
